# Medica: training and evaluation

Runs the repository's own `train.py` and `evaluate.py` on the held-out split. Attach the `dataset/` folder as a Kaggle Dataset, then run. See `readme.md` for the architecture and the measured scores.

In [ ]:
# --- Setup: get the repo into the notebook runtime -----------------------
# On Kaggle: add the repo as a "utility" input dataset, or set REPO_URL.
# The clone lands in /kaggle/working, which is the only writable area and
# what gets saved with the version.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/NickLitharis/medica.git"
WORK = Path("/kaggle/working")
REPO = WORK / "medica"

if not REPO.exists():
    try:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)
    except Exception as e:
        print("git clone failed:", e)
        print("Upload the repo as a Kaggle dataset and set REPO to that folder instead.")
    else:
        print("Cloned to", REPO)

if REPO.exists():
    sys.path.insert(0, str(REPO))
    os.chdir(REPO)
    print("Working directory:", os.getcwd())
else:
    print("Repo missing: the rest of this notebook cannot run.")


In [ ]:
# --- Resolve input and output paths --------------------------------------
# Attach the datasets first (right sidebar -> Add Data):
#   * the training dataset (the folder with image_features.pkl) - needed to train
#   * figure images + s2_cleaned.json - only for the optional rebuild stages
# None of them are found by name; the folders are searched by content.
from config.paths import resolve, check, report_dataset
paths = resolve()
check(paths)
report_dataset()   # finds the training dataset by its contents, whatever it is called


In [ ]:
# --- Stage 4: train -----------------------------------------------------
# Locates the dataset by content (no slug needed) and, if captions_fixed.pkl is
# there, trains on the complete captions. model_best.keras and encoding.json
# land in --out.
from config.paths import find_dataset_dir, dataset_dir_with_repaired_captions
dataset = dataset_dir_with_repaired_captions(find_dataset_dir())
print("training from:", dataset)

!python3 train.py --dataset-dir "$dataset"     --out /kaggle/working/medica/run --epochs 30 --batch-size 128 --arch seq2seq


In [ ]:
# --- Stage 5: score on the held-out split -------------------------------
!python3 evaluate.py --dataset-dir "$dataset"     --run /kaggle/working/medica/run --beam-width 1


In [ ]:
# --- Stage 6: a few generated captions, side by side --------------------
import json
from pathlib import Path
import tensorflow as tf
import pandas as pd

run = Path("/kaggle/working/medica/run")
enc = json.load(open(run / "encoding.json"))
print("run configuration:", {k: enc[k] for k in list(enc)[:8]})
print("artifacts:", sorted(p.name for p in run.iterdir()))
